# 02 — Sample Selection

Apply quality cuts to the APOGEE DR17 allStarLite catalog to define a working sample for binary star detection. Cuts include S/N, number of visits, STARFLAG, effective temperature, and surface gravity filters.

In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Add project root to path
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from config import LABEL_CONFIG
from src.data import load_allstar_catalog, filter_allstar_catalog

%matplotlib inline

## 1. Load Catalog

In [ ]:
catalog = load_allstar_catalog()
print(f"Full allStarLite catalog: {len(catalog)} stars, {len(catalog.colnames)} columns")

## 2. Apply Quality Cuts

Sequential cuts applied to the catalog:
1. **S/N ≥ 50** — minimum combined signal-to-noise ratio
2. **NVISITS ≥ 3** — enough visits for reliable RV scatter
3. **STARFLAG** — remove stars with STAR_BAD flag (bit 23)
4. **Teff ∈ [3500, 7000] K** — restrict to FGK/early-M dwarfs and giants
5. **log g ∈ [−0.5, 5.0]** — exclude unreliable surface gravity values

In [ ]:
filtered, waterfall = filter_allstar_catalog(catalog)

# Print waterfall table
cut_labels = [
    ("initial",      "Initial catalog"),
    ("snr_cut",      f"S/N >= {LABEL_CONFIG['snr_min']}"),
    ("nvisits_cut",  f"NVISITS >= {LABEL_CONFIG['min_visits']}"),
    ("starflag_cut", "STAR_BAD flag removed"),
    ("teff_cut",     f"Teff in {LABEL_CONFIG['teff_range']} K"),
    ("logg_cut",     f"log g in {LABEL_CONFIG['logg_range']}"),
    ("final",        "Final sample"),
]

print(f"{'Cut':<30s} {'Count':>10s} {'% of initial':>14s}")
print("-" * 56)
n_init = waterfall["initial"]
for key, label in cut_labels:
    count = waterfall[key]
    pct = 100.0 * count / n_init
    print(f"{label:<30s} {count:>10,d} {pct:>13.1f}%")

## 3. Filtered Sample Properties

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

# Top-left: Teff
ax = axes[0, 0]
ax.hist(filtered["TEFF"], bins=80, color="steelblue", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Teff [K]")
ax.set_ylabel("Count")
ax.set_title("Effective Temperature")
ax.invert_xaxis()

# Top-right: logg
ax = axes[0, 1]
ax.hist(filtered["LOGG"], bins=80, color="darkorange", edgecolor="white", linewidth=0.3)
ax.set_xlabel("log g [dex]")
ax.set_ylabel("Count")
ax.set_title("Surface Gravity")

# Bottom-left: [Fe/H]
ax = axes[1, 0]
ax.hist(filtered["FE_H"], bins=80, color="seagreen", edgecolor="white", linewidth=0.3)
ax.set_xlabel("[Fe/H] [dex]")
ax.set_ylabel("Count")
ax.set_title("Metallicity")

# Bottom-right: SNR
ax = axes[1, 1]
ax.hist(filtered["SNR"], bins=80, color="indianred", edgecolor="white", linewidth=0.3)
ax.set_xlabel("Combined S/N")
ax.set_ylabel("Count")
ax.set_title("Signal-to-Noise Ratio")

plt.tight_layout()
os.makedirs(os.path.join(project_root, "figures"), exist_ok=True)
fig.savefig(os.path.join(project_root, "figures", "filtered_sample_properties.png"), dpi=150)
plt.show()
print("Saved to figures/filtered_sample_properties.png")

## 4. VSCATTER in the Filtered Sample

VSCATTER (radial-velocity scatter across visits) is the primary observable used to generate binary/single labels. Stars with high VSCATTER are binary candidates; stars with low VSCATTER are likely single.

In [ ]:
vs = filtered["VSCATTER"]
vs_thresh_binary = LABEL_CONFIG["rv_scatter_binary_threshold"]
vs_thresh_single = LABEL_CONFIG["rv_scatter_single_threshold"]

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(vs, bins=200, color="slategray", edgecolor="white", linewidth=0.3, log=True)
ax.axvline(vs_thresh_single, color="blue", ls="--", lw=1.5, label=f"Single threshold ({vs_thresh_single} km/s)")
ax.axvline(vs_thresh_binary, color="red", ls="--", lw=1.5, label=f"Binary threshold ({vs_thresh_binary} km/s)")
ax.set_xlabel("VSCATTER [km/s]")
ax.set_ylabel("Count (log scale)")
ax.set_title("RV Scatter Distribution — Filtered Sample")
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(project_root, "figures", "vscatter_distribution.png"), dpi=150)
plt.show()

# Counts in each zone
n_single = np.sum(vs < vs_thresh_single)
n_ambiguous = np.sum((vs >= vs_thresh_single) & (vs <= vs_thresh_binary))
n_binary = np.sum(vs > vs_thresh_binary)
n_total = len(vs)

print(f"\nVSCATTER classification zones (N = {n_total:,d}):")
print(f"  Single    (VS < {vs_thresh_single} km/s):      {n_single:>8,d}  ({100*n_single/n_total:.1f}%)")
print(f"  Ambiguous ({vs_thresh_single}–{vs_thresh_binary} km/s):   {n_ambiguous:>8,d}  ({100*n_ambiguous/n_total:.1f}%)")
print(f"  Binary    (VS > {vs_thresh_binary} km/s):      {n_binary:>8,d}  ({100*n_binary/n_total:.1f}%)")
print(f"\nExpected class balance (excluding ambiguous):")
print(f"  Single : Binary = {n_single} : {n_binary} = 1 : {n_binary/max(n_single,1):.3f}")

## 5. Save Filtered Catalog

In [ ]:
output_path = os.path.join(project_root, "data", "allstar_filtered.fits")
os.makedirs(os.path.dirname(output_path), exist_ok=True)
filtered.write(output_path, format="fits", overwrite=True)

file_size_mb = os.path.getsize(output_path) / 1e6
print(f"Saved filtered catalog: {len(filtered):,d} stars")
print(f"Output: {output_path}")
print(f"File size: {file_size_mb:.1f} MB")